## Imports and paths

In [1]:
from pathlib import Path
import re
import csv
import codecs
import pandas as pd
import numpy as np
from IPython.display import display

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 150)

PROJECT_DIR = Path.cwd()

# The raw dataset path
DATA_DIR = PROJECT_DIR / "Data"
YOUGOV_DIR = DATA_DIR / "YouGov_Data"
OXCGRT_DIR = DATA_DIR / "OxCGRT_Data"

# Create the output folder
OUTPUT_DIR = PROJECT_DIR / "Cleaned Data"
FINAL_DIR = OUTPUT_DIR / "Uniform column format"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

## 1 Read data

In [3]:
# 1. Country and file settings

COUNTRY_NAMES = {"AUS": "Australia",  "BRA": "Brazil",  "CAN": "Canada",  "CHN": "China",  "GBR": "United Kingdom",  "IND": "India",  "USA": "United States"}

DATA_FILES = [
    ("OxCGRT", "AUS", OXCGRT_DIR / "OxCGRT_AUS_latest.csv"),
    ("OxCGRT", "BRA", OXCGRT_DIR / "OxCGRT_BRA_latest LEGACY.csv"),
    ("OxCGRT", "CAN", OXCGRT_DIR / "OxCGRT_CAN_latest.csv"),
    ("OxCGRT", "CHN", OXCGRT_DIR / "OxCGRT_CHN_latest.csv"),
    ("OxCGRT", "GBR", OXCGRT_DIR / "OxCGRT_GBR_latest.csv"),
    ("OxCGRT", "IND", OXCGRT_DIR / "OxCGRT_IND_latest.csv"),
    ("OxCGRT", "USA", OXCGRT_DIR / "OxCGRT_USA_latest.csv"),

    ("YouGov", "AUS", YOUGOV_DIR / "australia.csv"),
    ("YouGov", "BRA", YOUGOV_DIR / "brazil.csv"),
    ("YouGov", "CAN", YOUGOV_DIR / "canada.csv"),
    ("YouGov", "CHN", YOUGOV_DIR / "china.csv"),
    ("YouGov", "GBR", YOUGOV_DIR / "united-kingdom.csv"),
    ("YouGov", "IND", YOUGOV_DIR / "india.csv"),
    ("YouGov", "USA", YOUGOV_DIR / "united-states.csv")]

SAMPLE_ROWS = 5000
ENCODINGS = ["utf-8-sig", "utf-8", "cp1252", "latin1"]


# 2. Read all CSV 
def read_csv_sample(path):
    for encoding in ENCODINGS:
        try:
            df = pd.read_csv(path,encoding=encoding,nrows=SAMPLE_ROWS,na_values=[" ", "__NA__"],low_memory=False)
            return df, encoding
        except UnicodeDecodeError:
            continue
        except Exception:
            return None, encoding
    return None, None

rows = []
raw_data = {}

for source, country_code, path in DATA_FILES:
    df, encoding = read_csv_sample(path)
    raw_data[(source, country_code)] = df
    column_count = df.shape[1]
    rows.append({"source": source,  "country_code": country_code,  "country_name": COUNTRY_NAMES[country_code],  "filename": path.name,  "encoding": encoding,  "column_count": column_count})

encoding_check = pd.DataFrame(rows)



# 3. Format check

format_issues = encoding_check[(~encoding_check["encoding"].isin(["utf-8-sig", "utf-8"]))|(encoding_check["column_count"].isna())].copy()

print("Data Format:")
if format_issues.empty:
    display(pd.DataFrame({"message": ["Everything OK"]}))
else:
    display(format_issues)
display(encoding_check)

Data Format:


,source,country_code,country_name,filename,encoding,column_count
11,YouGov,GBR,United Kingdom,united-kingdom.csv,cp1252,528


,source,country_code,country_name,filename,encoding,column_count
0,OxCGRT,AUS,Australia,OxCGRT_AUS_latest.csv,utf-8-sig,61
1,OxCGRT,BRA,Brazil,OxCGRT_BRA_latest LEGACY.csv,utf-8-sig,61
2,OxCGRT,CAN,Canada,OxCGRT_CAN_latest.csv,utf-8-sig,61
3,OxCGRT,CHN,China,OxCGRT_CHN_latest.csv,utf-8-sig,61
4,OxCGRT,GBR,United Kingdom,OxCGRT_GBR_latest.csv,utf-8-sig,61
5,OxCGRT,IND,India,OxCGRT_IND_latest.csv,utf-8-sig,61
6,OxCGRT,USA,United States,OxCGRT_USA_latest.csv,utf-8-sig,61
7,YouGov,AUS,Australia,australia.csv,utf-8-sig,513
8,YouGov,BRA,Brazil,brazil.csv,utf-8-sig,310
9,YouGov,CAN,Canada,canada.csv,utf-8-sig,512


## 2 Check key columns

### 2.1 All key columns

In [4]:
# key columns
USED_OXCGRT_COLS = ["RegionName","RegionCode","Date","H6M_Facial Coverings"]  # OxCGRT key columns
BASE_YOUGOV_COLS = ["RecordNo","endtime","i2_health","i9_health","i11_health","age","gender","state","household_size","employment_status","cantril_ladder","WCRex1","WCRex2","r1_1","r1_2","PHQ4_1","PHQ4_2","PHQ4_3","PHQ4_4"]  # Yougov kay columns
D1_COLS = [f"d1_health_{i}" for i in range(1, 14)] + ["d1_health_98","d1_health_99"]

# Read Australia YouGov dataset
aus_row = encoding_check[(encoding_check["source"] == "YouGov") &(encoding_check["country_code"] == "AUS")].iloc[0]
aus_path = [path for source, country_code, path in DATA_FILES if source == "YouGov" and country_code == "AUS"][0]
aus = pd.read_csv(aus_path,encoding=aus_row["encoding"],sep=",",na_values=[" ", "__NA__"],low_memory=False)

# Keep i12 columns with missing rate <= 30%
aus_missing_rate = aus.isna().mean()
aus_retained_cols = aus_missing_rate[aus_missing_rate <= 0.30].index.tolist()

I12_COLS = [c for c in aus_retained_cols if str(c).startswith("i12_")]

# Combine key YouGov columns
USED_YOUGOV_COLS = BASE_YOUGOV_COLS + D1_COLS + I12_COLS

# Visualise
used_columns = pd.DataFrame([{"source": "OxCGRT", "column": col} for col in USED_OXCGRT_COLS] +[{"source": "YouGov", "column": col} for col in USED_YOUGOV_COLS])
display(used_columns)
print("OxCGRT key columns:", len(USED_OXCGRT_COLS))
print("YouGov key columns:", len(USED_YOUGOV_COLS))

,source,column
0,OxCGRT,RegionName
1,OxCGRT,RegionCode
2,OxCGRT,Date
3,OxCGRT,H6M_Facial Coverings
4,YouGov,RecordNo
5,YouGov,endtime
6,YouGov,i2_health
7,YouGov,i9_health
8,YouGov,i11_health
9,YouGov,age


OxCGRT key columns: 4
YouGov key columns: 52


### 2.2 Check whether all countries contain all key columns

In [5]:
def check_missing_columns(data_dict, source, columns):
    rows = []
    for country in COUNTRY_NAMES:
        key = (source, country)
        if key not in data_dict:
            rows.append({"source": source,"country_code": country,"country_name": COUNTRY_NAMES[country],"column": None,"problem": "file not found"})
            continue
        df = data_dict[key]
        for col in columns:
            if col not in df.columns:
                rows.append({"source": source,"country_code": country,"country_name": COUNTRY_NAMES[country],"column": col,"problem": "missing column"})
    return pd.DataFrame(rows)

# Check if contain key columns
raw_missing = pd.concat([check_missing_columns(raw_data, "OxCGRT", USED_OXCGRT_COLS),check_missing_columns(raw_data, "YouGov", USED_YOUGOV_COLS),],ignore_index=True)

# Visualise
print("Errors:")
if raw_missing.empty:
    display(pd.DataFrame({"message": ["No missing columns"]}))
else:
    display(raw_missing)

Errors:


,source,country_code,country_name,column,problem
0,OxCGRT,BRA,Brazil,H6M_Facial Coverings,missing column
1,YouGov,BRA,Brazil,state,missing column
2,YouGov,CAN,Canada,state,missing column
3,YouGov,CHN,China,state,missing column
4,YouGov,CHN,China,employment_status,missing column
5,YouGov,CHN,China,WCRex1,missing column
6,YouGov,CHN,China,WCRex2,missing column
7,YouGov,GBR,United Kingdom,state,missing column


## 3 Uniform column format

In [6]:
# Rename columns
RENAME_MAP = {
    ("OxCGRT", "BRA"): {"H6_Facial Coverings": "H6M_Facial Coverings"},
    ("YouGov", "BRA"): {"region": "state"},
    ("YouGov", "CAN"): {"region": "state"},
    ("YouGov", "GBR"): {"region": "state"}}

# Delete China
EXCLUDED_COUNTRIES = ["CHN"]
FINAL_COUNTRIES = [country for country in COUNTRY_NAMES if country not in EXCLUDED_COUNTRIES]


# Read new datasets
uniform_data = {}
for (source, country), df in raw_data.items():
    if country in EXCLUDED_COUNTRIES:
        continue
    df = df.copy()
    df = df.rename(columns=RENAME_MAP.get((source, country), {}))
    uniform_data[(source, country)] = df


# Visualise operations
operation_log = pd.DataFrame([
    {"operation": "rename column",  "source": "OxCGRT",  "country_code": "BRA",  "country_name": COUNTRY_NAMES["BRA"],  "original_column": "H6_Facial Coverings",  "new_column": "H6M_Facial Coverings"},
    {"operation": "rename column",  "source": "YouGov",  "country_code": "BRA",  "country_name": COUNTRY_NAMES["BRA"],  "original_column": "region",  "new_column": "state"},
    {"operation": "rename column",  "source": "YouGov",  "country_code": "CAN",  "country_name": COUNTRY_NAMES["CAN"],  "original_column": "region",  "new_column": "state"},
    {"operation": "rename column",  "source": "YouGov",  "country_code": "GBR",  "country_name": COUNTRY_NAMES["GBR"],  "original_column": "region",  "new_column": "state"},
    {"operation": "exclude country",  "source": "All",  "country_code": "CHN",  "country_name": COUNTRY_NAMES["CHN"],  "original_column": None,  "new_column": None}])
display(operation_log)

print("Excluded countries:", EXCLUDED_COUNTRIES)
print("Included countries:", FINAL_COUNTRIES)

,operation,source,country_code,country_name,original_column,new_column
0,rename column,OxCGRT,BRA,Brazil,H6_Facial Coverings,H6M_Facial Coverings
1,rename column,YouGov,BRA,Brazil,region,state
2,rename column,YouGov,CAN,Canada,region,state
3,rename column,YouGov,GBR,United Kingdom,region,state
4,exclude country,All,CHN,China,NaN,NaN


Excluded countries: ['CHN']
Included countries: ['AUS', 'BRA', 'CAN', 'GBR', 'IND', 'USA']


## 4 Save files

In [7]:
FINAL_DIR.mkdir(parents=True, exist_ok=True)

encoding_map = encoding_check.set_index(["source", "country_code"])["encoding"].to_dict()
saved_rows = []

for source, country, path in DATA_FILES:
    if country in EXCLUDED_COUNTRIES:
        continue
    
    # Read
    df = pd.read_csv(path,encoding=encoding_map[(source, country)],na_values=[" ", "__NA__"],low_memory=False).rename(columns=RENAME_MAP.get((source, country), {}))

    # Save file
    output_name = f"{source}_{COUNTRY_NAMES[country]}.csv"
    output_path = FINAL_DIR / output_name
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    # Visualise 1
    saved_rows.append({ "source": source,  "country_code": country,  "country_name": COUNTRY_NAMES[country],  "saved_file": output_name,  "rows": len(df),  "columns": df.shape[1]})

# Visualise 2
saved_summary = pd.DataFrame(saved_rows)
display(saved_summary)

,source,country_code,country_name,saved_file,rows,columns
0,OxCGRT,AUS,Australia,OxCGRT_Australia.csv,9864,61
1,OxCGRT,BRA,Brazil,OxCGRT_Brazil.csv,30688,61
2,OxCGRT,CAN,Canada,OxCGRT_Canada.csv,15344,61
3,OxCGRT,GBR,United Kingdom,OxCGRT_United Kingdom.csv,5480,61
4,OxCGRT,IND,India,OxCGRT_India.csv,40552,61
5,OxCGRT,USA,United States,OxCGRT_United States.csv,56992,61
6,YouGov,AUS,Australia,YouGov_Australia.csv,53833,513
7,YouGov,BRA,Brazil,YouGov_Brazil.csv,10308,310
8,YouGov,CAN,Canada,YouGov_Canada.csv,48372,512
9,YouGov,GBR,United Kingdom,YouGov_United Kingdom.csv,64532,528
